In [1]:
!pip install fastapi uvicorn pyngrok transformers==4.52.4 accelerate -q
!pip install faiss-cpu langchain langchain-community langchain-classic langchain-core langchain-huggingface pypdf sentence-transformers torch

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import torch
import re
import json

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
).eval()

def generate_text(prompt, max_length=2048):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
response_schemas = [
    ResponseSchema(
        name="full_name",
        description="Candidate's full name extracted from CV",
        type="string"
    ),
    ResponseSchema(
        name="email",
        description="Candidate's email address",
        type="string"
    ),
    ResponseSchema(
        name="education",
        description="List of educational qualifications (degree, field, institution, year)",
        type="List[dict]"
    ),
    ResponseSchema(
        name="skills",
        description="List of all skills explicitly demonstrated in the CV",
        type="List[string]"
    ),
    ResponseSchema(
        name="experience",
        description="List of work experience (role, company, dates, responsibilities)",
        type="List[dict]"
    ),
    ResponseSchema(
        name="career_recommendation",
        description="""Detailed career recommendation object containing:
        - career_name: string (Name of the top recommended career)
        - match_score: string (Estimated percentage match, e.g., '85%')
        - reason: string (Detailed explanation of why this career is the best fit)
        - current_relevant_skills: List[string] (Existing skills directly relevant to this career)
        - missing_skills: List[string] (Required skills not demonstrated in the CV)
        - skill_priorities: List[string] (Missing skills ordered from highest to lowest priority)
        - roadmap: List[string] (Step-by-step ordered list of actionable items to gain missing skills)
        - suggested_projects: List[string] (Practical project ideas to demonstrate readiness)
        - suggested_job_titles: List[string] (Entry/Mid level job titles to search for)""",
        type="dict"
    )
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()

info_extraction_template = """
You are an expert AI Career Coach analyzing a candidate's CV to recommend their single best career path.

Your task is to analyze the user's CV, summarize their existing profile, identify the single most suitable career path, highlight skill gaps, and provide a structured, step-by-step roadmap.

### Step 1 — Extract Career Profile
Extract explicitly mentioned information from the CV:
* Full Name & Email
* Educational qualifications
* Skills (Technical, Soft, Tools, Languages)
* Work & Internship Experience

Only use information explicitly stated in the CV. Do not assume or fabricate credentials.

### Step 2 — Identify the Best Career Path
Select the single closest matching career based on their education, existing skills, and past experience.

### Step 3 — Analyze Current vs. Missing Skills
* Identify relevant skills the candidate already possesses.
* Identify missing or non-demonstrated skills required for the target role.
* Rank missing skills by learning priority.

### Step 4 — Generate Actionable Roadmap
Provide a practical, ordered list of actionable steps (as concise bullet points) for the candidate to bridge their skill gaps and become job-ready.

### Constraints
* Base analysis solely on provided text.
* Return ONLY valid JSON matching the format instructions. Do not include markdown preamble outside the JSON output.

{format_instructions}

CV Input:
{user_input}
"""

In [4]:
prompt = PromptTemplate(
    template=info_extraction_template,
    input_variables=["user_input"],
    partial_variables={"format_instructions": format_instructions},
)


In [ ]:
NGROK_TOKEN = "NGROCK_TOKEN"  # Replace with your actual ngrok token
API_KEY = "API_KEY"  # Replace with your actual API key

In [6]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf

app = FastAPI()

@app.post("/generate")
async def gen(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    data = await req.json()
    sample_input = data.get("prompt", "")
    formatted_prompt_str = prompt.format(user_input=sample_input)
    messages = [{"role": "user", "content": formatted_prompt_str}]
    formatted_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    return {
        "response": generate_text(
            formatted_chat,
            data.get("max_length", 2048)
        )
    }

In [7]:
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

Your public URL: https://detergent-direction-wrongness.ngrok-free.dev


INFO:     Started server process [267]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:56465 (Press CTRL+C to quit)


INFO:     197.36.133.229:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.133.229:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.133.229:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.133.229:0 - "POST /generate HTTP/1.1" 200 OK
